# Deploy the base model

This tutorial shows you how to deploy a base model on Modal to use for evaluation. Here, we will use Qwen3-4B as the base model, but
you can use any other model that is supported by the training-gym, or configure your own.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import re

from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HuggingFaceDataset,
    Qwen3_4B,
    SlimeRecipe,
)

## Serve the base model

To deploy the base model, we can use the `DeploymentConfig` class. This allows us to deploy the model on Modal and get a handle to the deployed model.

In [ ]:
base_model = Qwen3_4B()
base_model_deployment = DeploymentConfig(
    model=base_model,
).serve()
print(f"Base model deployed to {base_model_deployment.url}")

Now that the model has come alive, we can request it to generate text.

Note that this is creating an SGLang server app and starting it up, then sending a request, so the first request should take a few minutes to complete.

In [ ]:
response = base_model_deployment.generate(
    "Write a short story about a cat.",
)
print(response)

The `chat_template_kwargs` parameter allows us to pass in a dictionary of kwargs to the chat template.
This is useful for passing in additional parameters to the chat template, such as the enable_thinking parameter.

In [ ]:
response = base_model_deployment.generate(
    "Write a short story about a cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)

In [ ]:
"""
Remember the training gym dashboard you deployed in the previous tutorial?
We can now see the base model deployment in the dashboard.
"""

In [ ]:
print(base_model_deployment.modal_app_url)